Chirag Bansal

In [5]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

class GaussianNB_Scratch:
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)

        # Initialize mean, variance, and priors for each class
        self.mean = np.zeros((n_classes, n_features))
        self.var = np.zeros((n_classes, n_features))
        self.priors = np.zeros(n_classes)

        for idx, c in enumerate(self.classes):
            # Filter samples for specific class
            X_c = X[y == c]

            # Calculate Mean, Variance, and Prior Probability
            self.mean[idx, :] = X_c.mean(axis=0)
            self.var[idx, :] = X_c.var(axis=0)
            self.priors[idx] = X_c.shape[0] / float(n_samples)

    def _gaussian_density(self, class_idx, x):
        mean = self.mean[class_idx]
        var = self.var[class_idx]
        numerator = np.exp(-((x - mean) ** 2) / (2 * var))
        denominator = np.sqrt(2 * np.pi * var)
        return numerator / denominator

    def predict(self, X):
        y_pred = [self._predict_single(x) for x in X]
        return np.array(y_pred)

    def _predict_single(self, x):
        posteriors = []

        for idx, c in enumerate(self.classes):
            # Prior P(y)
            prior = np.log(self.priors[idx])

            # Likelihood P(X|y)
            # We use Log-sum-exp trick for numerical stability (sum of logs instead of product of probs)
            posterior = np.sum(np.log(self._gaussian_density(idx, x)))
            posterior = prior + posterior
            posteriors.append(posterior)

        return self.classes[np.argmax(posteriors)]

# 1. Load Data
data = load_iris()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Train Scratch Model
nb_scratch = GaussianNB_Scratch()
nb_scratch.fit(X_train, y_train)

# 3. Evaluate
y_pred_scratch = nb_scratch.predict(X_test)

print("--- Step-by-Step Implementation Results ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_scratch):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_scratch))

--- Step-by-Step Implementation Results ---
Accuracy: 1.0000

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



In [6]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load Dataset (Breast Cancer)
data = load_breast_cancer()
X = data.data
y = data.target

# 2. Preprocessing (Standardization is crucial for KNN)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

# 3. Define the Model
knn = KNeighborsClassifier()

# 4. Define the Parameter Grid
# We will search for the best 'n_neighbors' (K) and 'weights' (uniform vs distance)
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 13, 15, 21],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# 5. Initialize GridSearchCV
# cv=5 means 5-fold cross-validation
grid_search = GridSearchCV(estimator=knn, param_grid=param_grid, cv=5, scoring='accuracy', verbose=1)

# 6. Perform the Search
print("Starting Grid Search...")
grid_search.fit(X_train, y_train)

# 7. Results
print("\n--- Grid Search Results ---")
print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Best Cross-Validation Score: {grid_search.best_score_:.4f}")

# 8. Evaluate best model on hold-out Test Set
best_model = grid_search.best_estimator_
test_accuracy = best_model.score(X_test, y_test)
print(f"Test Set Accuracy with Best Params: {test_accuracy:.4f}")

Starting Grid Search...
Fitting 5 folds for each of 32 candidates, totalling 160 fits

--- Grid Search Results ---
Best Parameters Found: {'metric': 'manhattan', 'n_neighbors': 3, 'weights': 'uniform'}
Best Cross-Validation Score: 0.9623
Test Set Accuracy with Best Params: 0.9708
